# Manga Image Translator - Google Colab

Translate manga/images into your desired language using AI.

**⚠️ Make sure to select a GPU runtime before running:**
`Runtime` → `Change runtime type` → Select `T4 GPU` (or better)

In [ ]:
# Clone the repository (fork with Load Text / Export OCR features)
!git clone https://github.com/mugnimaestra/manga-image-translator
!cd manga-image-translator && git checkout feature/web-ui-load-export-text
%cd /content/manga-image-translator/

# ==============================================================================
# 📦 Install Dependencies (Colab T4 GPU)
# ==============================================================================
# Colab pre-installs its own torch (e.g., 2.6.0+cu126) which does NOT include
# the nvidia pip packages (cublas, cudnn, cusparse, cusparselt, etc.) that
# PyTorch dynamically links against. This causes:
#   ImportError: libcusparseLt.so.0: cannot open shared object file
#
# Fix: UNINSTALL Colab's torch first → install from cu124 index (which bundles
# all 13 nvidia dependencies) → then install project requirements.
# ==============================================================================

import subprocess, sys

# Step 1: Remove Colab's pre-installed torch (which lacks nvidia deps)
print("\ud83d\udd27 Step 1: Removing Colab's pre-installed PyTorch...")
subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y',
                       'torch', 'torchvision', 'torchaudio'])
print("   \u2705 Old torch removed\n")

# Step 2: Install CUDA-enabled PyTorch from cu124 index (includes all nvidia libs)
print("\ud83d\udd27 Step 2: Installing PyTorch with CUDA 12.4 support...")
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       'torch', 'torchvision',
                       '--index-url', 'https://download.pytorch.org/whl/cu124'])
print("   \u2705 CUDA PyTorch installed\n")

# Step 3: Install project requirements (torch already satisfied → no overwrite)
print("\ud83d\udd27 Step 3: Installing project requirements...")
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'])
print("   \u2705 Requirements installed\n")

# Step 4: Verify torch imports correctly in a subprocess
# (subprocess doesn't pollute the notebook kernel's cached module state)
print("\ud83d\udd27 Step 4: Verifying PyTorch CUDA in subprocess...")
result = subprocess.run(
    [sys.executable, '-c',
     'import torch; print(f"torch={torch.__version__}"); '
     'print(f"cuda={torch.cuda.is_available()}"); '
     'print(f"gpu={torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}")'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0 or 'cuda=False' in result.stdout:
    print("\u26a0\ufe0f CUDA verification failed. Stderr:")
    print(result.stderr)
    print("\n\ud83d\udd04 Attempting LD_LIBRARY_PATH fix...")
    # Find nvidia lib dirs and add to LD_LIBRARY_PATH
    import os, glob as g
    nvidia_dirs = g.glob(f'{sys.prefix}/lib/python*/site-packages/nvidia/*/lib')
    if nvidia_dirs:
        ld_path = ':'.join(nvidia_dirs)
        os.environ['LD_LIBRARY_PATH'] = ld_path + ':' + os.environ.get('LD_LIBRARY_PATH', '')
        print(f"   Set LD_LIBRARY_PATH with {len(nvidia_dirs)} nvidia dirs")
        # Re-verify
        env = os.environ.copy()
        env['LD_LIBRARY_PATH'] = ld_path + ':' + env.get('LD_LIBRARY_PATH', '')
        result2 = subprocess.run(
            [sys.executable, '-c', 'import torch; print(torch.cuda.is_available())'],
            capture_output=True, text=True, env=env
        )
        if 'True' in result2.stdout:
            print("   \u2705 LD_LIBRARY_PATH fix worked!")
        else:
            print("   \u274c Still failing. You may need to restart the runtime.")
            print("   Go to: Runtime \u2192 Restart runtime, then re-run from this cell.")
    else:
        print("   \u274c No nvidia lib dirs found. Try restarting runtime.")
        print("   Go to: Runtime \u2192 Restart runtime, then re-run from this cell.")
else:
    print("\u2705 Installation complete! PyTorch CUDA is working.")

In [ ]:
# Verify CUDA is available
# Belt-and-suspenders: set LD_LIBRARY_PATH for nvidia pip packages before importing torch
import os, sys
try:
    import importlib
    # Find nvidia lib directories from pip-installed packages
    nvidia_lib_dirs = []
    site_pkgs = [p for p in sys.path if 'site-packages' in p]
    for sp in site_pkgs:
        nvidia_base = os.path.join(sp, 'nvidia')
        if os.path.isdir(nvidia_base):
            for sub in os.listdir(nvidia_base):
                lib_dir = os.path.join(nvidia_base, sub, 'lib')
                if os.path.isdir(lib_dir):
                    nvidia_lib_dirs.append(lib_dir)
    if nvidia_lib_dirs:
        existing = os.environ.get('LD_LIBRARY_PATH', '')
        os.environ['LD_LIBRARY_PATH'] = ':'.join(nvidia_lib_dirs) + (':' + existing if existing else '')
        # Also add to ctypes search path for already-loaded process
        import ctypes
        for d in nvidia_lib_dirs:
            try:
                for so_file in os.listdir(d):
                    if so_file.endswith('.so') or '.so.' in so_file:
                        try:
                            ctypes.CDLL(os.path.join(d, so_file), mode=ctypes.RTLD_GLOBAL)
                        except OSError:
                            pass
            except Exception:
                pass
except Exception as e:
    print(f'\u26a0\ufe0f LD_LIBRARY_PATH setup warning: {e}')

import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')
    print('\u2705 Ready to use!')
else:
    print('\u274c CUDA not available! Try these fixes:')
    print('  1. Make sure you selected a GPU runtime: Runtime \u2192 Change runtime type \u2192 T4 GPU')
    print('  2. Restart the runtime: Runtime \u2192 Restart runtime')
    print('  3. Re-run the install cell above')

## (Optional) Set translator API keys

If you want to use cloud-based translators (e.g., GPT, DeepL, Gemini), set the corresponding
environment variables below. Skip this cell if you only need the default translator.

In [ ]:
# Uncomment and fill in the API keys for the translators you want to use
import os

# OpenAI / ChatGPT
# os.environ['OPENAI_API_KEY'] = 'your-key-here'
# os.environ['OPENAI_MODEL'] = 'gpt-4o-mini'

# DeepL
# os.environ['DEEPL_AUTH_KEY'] = 'your-key-here'

# Google Gemini
# os.environ['GOOGLE_GEMINI_API_KEY'] = 'your-key-here'

## Option A: Web UI Mode

Run the web server and use the browser-based interface. Supports Load Text and Export OCR Text features.

In [ ]:
# Start the web server with Colab proxy
from google.colab.output import eval_js

port = 8000
url = eval_js(f"google.colab.kernel.proxyPort({port})")
print(f'Open this link to use manga-image-translator → {url}')

!python server/main.py --host 0.0.0.0 --port {port} --use-gpu --verbose

## Option B: CLI Batch Mode - OCR Export → Manual Edit → Load Text

A 3-step workflow for manual translation control:
1. **Export OCR** — detect text and save raw OCR as JSON
2. **Edit** — manually translate/fix the text in the JSON file
3. **Load + Render** — load your translations and render the final image

This is optimized for T4 GPU (16 GB VRAM).

In [ ]:
# Upload your manga images
import os
from google.colab import files

upload_dir = '/content/manga_input'
os.makedirs(upload_dir, exist_ok=True)

print('Upload your manga images:')
uploaded = files.upload()
for filename in uploaded:
    dest = os.path.join(upload_dir, filename)
    with open(dest, 'wb') as f:
        f.write(uploaded[filename])
    print(f'  Saved: {dest}')

print(f'\n✅ {len(uploaded)} image(s) uploaded to {upload_dir}')

In [ ]:
# Create T4-optimized config files
import json

# Config for Step 1: Export OCR only (lightweight — no inpainter loaded)
export_config = {
    "detector": {
        "detector": "default",
        "detection_size": 1536,
        "text_threshold": 0.5,
        "box_threshold": 0.7,
        "unclip_ratio": 2.3
    },
    "ocr": {
        "ocr": "48px",
        "min_text_length": 0
    },
    "translator": {
        "translator": "none",
        "target_lang": "ENG"
    },
    "inpainter": {
        "inpainter": "none"
    },
    "upscale": {
        "upscaler": "esrgan",
        "upscale_ratio": None
    },
    "colorizer": {
        "colorizer": "none"
    }
}

# Config for Step 3: Load text + inpaint + render
render_config = {
    "detector": {
        "detector": "default",
        "detection_size": 1536,
        "text_threshold": 0.5,
        "box_threshold": 0.7,
        "unclip_ratio": 2.3
    },
    "ocr": {
        "ocr": "48px",
        "min_text_length": 0
    },
    "translator": {
        "translator": "none",
        "target_lang": "ENG"
    },
    "inpainter": {
        "inpainter": "lama_large",
        "inpainting_size": 1024,
        "inpainting_precision": "bf16"
    },
    "render": {
        "renderer": "manga2eng",
        "alignment": "auto",
        "direction": "auto",
        "font_size_offset": 0,
        "no_hyphenation": True
    },
    "upscale": {
        "upscaler": "esrgan",
        "upscale_ratio": None
    },
    "colorizer": {
        "colorizer": "none"
    },
    "mask_dilation_offset": 20,
    "kernel_size": 3
}

with open('/content/t4_export_ocr.json', 'w') as f:
    json.dump(export_config, f, indent=2)

with open('/content/t4_load_render.json', 'w') as f:
    json.dump(render_config, f, indent=2)

print('✅ T4-optimized config files created:')
print('  /content/t4_export_ocr.json   (Step 1: ~2-3 GB VRAM)')
print('  /content/t4_load_render.json  (Step 3: ~6-8 GB VRAM)')

### Step 1: Export Raw OCR Text

Runs detection + OCR → saves detected text as JSON → exits.
No translation or inpainting model is loaded.

In [ ]:
%cd /content/manga-image-translator

# Export raw OCR text (no translation, no inpainting loaded)
!python -m manga_translator local \
    --save-text \
    --use-gpu \
    -i /content/manga_input/ \
    --config-file /content/t4_export_ocr.json

In [ ]:
# Find and display the exported OCR text files
import glob, json

ocr_files = glob.glob('result/*_translations.txt') + glob.glob('results/*_translations.txt')
if not ocr_files:
    print('⚠️ No OCR text files found. Make sure Step 1 ran successfully.')
else:
    for f in ocr_files:
        print(f'📄 {f}')
        with open(f, 'r') as fh:
            data = json.load(fh)
        for i, text in enumerate(data):
            print(f'  [{i}] {text}')
        print()

    # Download for editing
    from google.colab import files
    for f in ocr_files:
        files.download(f)
    print('✅ Downloaded! Edit the JSON file and re-upload in the next step.')

### Step 2: Upload Edited Translations

Edit the downloaded JSON file — replace the OCR text with your translations.

The format is a simple JSON array:
```json
["Translation for bubble 1", "Translation for bubble 2", "..."]
```

Then upload it back:

In [ ]:
# Upload your edited translations file
import shutil, glob
from google.colab import files

print('Upload your edited _translations.txt file:')
uploaded = files.upload()

# Replace the original file with the edited version
existing_files = glob.glob('result/*_translations.txt') + glob.glob('results/*_translations.txt')
for filename in uploaded:
    if existing_files:
        dest = existing_files[0]  # Replace the first match
    else:
        dest = f'result/{filename}'
    with open(dest, 'wb') as f:
        f.write(uploaded[filename])
    print(f'✅ Saved edited translations to: {dest}')

### Step 3: Load Translations + Inpaint + Render

Loads your edited translations, runs inpainting to remove original text, and renders your translations onto the image.

In [ ]:
%cd /content/manga-image-translator

# Load edited translations + inpaint + render
!python -m manga_translator local \
    --load-text \
    --use-gpu \
    -i /content/manga_input/ \
    -o /content/manga_output/ \
    --config-file /content/t4_load_render.json

In [ ]:
# Display and download results
import glob
from IPython.display import display, Image
from google.colab import files

output_files = glob.glob('/content/manga_output/**/*.*', recursive=True)
output_files = [f for f in output_files if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))]

if not output_files:
    print('⚠️ No output images found.')
else:
    for f in output_files:
        print(f'🖼️ {f}')
        display(Image(filename=f, width=400))
        files.download(f)
    print(f'\n✅ {len(output_files)} image(s) downloaded!')